# Laboratorio 7 – Spark MLlib

## Avance: análisis exploratorio y segmentación

**Universidad:** Universidad del Valle de Guatemala  
**Curso:** CC3066 – Data Science  
**Estudiante:** Nery Molina  
**Carné:** 23218  
**Semestre:** II – 2026


## Objetivos

- Preparar y armonizar las bases ENEIC del módulo Personas.
- Evaluar la calidad de la información utilizada.
- Explorar la población asalariada seleccionada.
- Analizar asociaciones entre salario y variables laborales.
- Identificar perfiles de trabajadores mediante KMeans.

### Alcance del avance

Este notebook implementa únicamente las actividades 1–4: calidad de datos,
exploración, correlaciones y segmentación. Las actividades de modelos
supervisados y su evaluación final quedan fuera de este avance.

Los resultados empíricos de exploración, correlaciones y KMeans se calculan
solo si están disponibles los cuatro períodos de Personas de 2025. Personas
I-2026 se prepara por separado y no se incorpora a esos resultados.


## Configuración y reproducibilidad

La semilla se fija en 42. Las rutas se derivan desde la raíz del proyecto, por
lo que no dependen de una computadora específica. Antes de ejecutar, instale
las dependencias de requirements.txt y configure Java 17 para Spark 3.5.x.


In [ ]:
from pathlib import Path
import math
import os
import sys
import warnings

SEED = 42
warnings.filterwarnings("ignore", category=FutureWarning)

def locate_project_root(start: Path) -> Path:
    """Busca la raíz sin depender de __file__, que no existe en notebooks."""
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "No fue posible localizar la raíz del proyecto. "
        "Abra el notebook desde este repositorio."
    )

PROJECT_ROOT = locate_project_root(Path.cwd().resolve())
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
FIGURES = OUTPUTS / "figures"
TABLES = OUTPUTS / "tables"

for directory in (DATA_RAW, DATA_PROCESSED, FIGURES, TABLES):
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_RAW: {DATA_RAW}")
print(f"SEED: {SEED}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.stat import Correlation

from src.utils import (
    apply_filters_with_audit,
    descriptive_statistics,
    detect_period_from_filename,
    find_person_files,
    load_excel_to_spark,
    median_by_group,
    missing_summary,
    normalize_code,
    prepare_kmeans_features,
    read_excel_columns,
)

spark = (
    SparkSession.builder
    .appName("Laboratorio_7_Avance_Nery_Molina_23218")
    .config("spark.sql.session.timeZone", "America/Guatemala")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark: {spark.version}")
print(f"Python/Pandas: {pd.__version__}")


# 1. Carga, armonización y calidad de datos

Las funciones auxiliares leen cada Excel por separado con pandas y openpyxl,
convierten el subconjunto requerido a un DataFrame de Spark y realizan el
procesamiento analítico posterior con PySpark. Los archivos de Hogares,
Vivienda u otros módulos no se consideran.


In [ ]:
REQUIRED_COLUMNS = [
    "ANIO",
    "TRIMESTRE",
    "DOMINIO",
    "NUM_HOGAR",
    "FACTOR",
    "NUM_PERSONA",
    "P02A03",
    "P03A03A",
    "P05C07A",
    "P05C07B",
    "P05C16",
    "P05D01",
    "P05H01A",
    "OCUPADOS",
]

SOURCE_COLUMNS = REQUIRED_COLUMNS.copy()
EXPECTED_2025 = ("2025T1", "2025T2", "2025T3", "2025T4")

person_files = find_person_files(DATA_RAW)
files_by_period = {}

for path in person_files:
    metadata = detect_period_from_filename(path)
    period = metadata["periodo_archivo"]
    if period in files_by_period:
        previous = files_by_period[period]
        raise ValueError(
            f"Se detectaron dos archivos Personas para {period}: "
            f"{previous.name} y {path.name}. Conserve solo uno."
        )
    files_by_period[period] = path

inventory_rows = []
for period, path in sorted(files_by_period.items()):
    metadata = detect_period_from_filename(path)
    inventory_rows.append(
        {
            "periodo_archivo": period,
            "archivo": path.name,
            "anio_archivo": metadata["anio_archivo"],
            "trimestre_calendario": metadata["trimestre_calendario"],
        }
    )

if inventory_rows:
    display(pd.DataFrame(inventory_rows))
else:
    print(
        "No se encontraron archivos de Personas en data/raw. "
        "Coloque allí los Excel ENEIC del módulo Personas."
    )


In [ ]:
i2026_path = files_by_period.get("2026T1")

if i2026_path is not None:
    i2026_sheet = "Personas_ENEIC_T1-2026"
    i2026_columns = read_excel_columns(i2026_path, i2026_sheet)
    missing_i2026 = sorted(set(REQUIRED_COLUMNS) - set(i2026_columns))

    if missing_i2026:
        available_preview = ", ".join(i2026_columns[:40])
        raise ValueError(
            f"El archivo {i2026_path.name} no contiene las columnas requeridas: "
            f"{missing_i2026}. Columnas disponibles (muestra): {available_preview}"
        )

    print(
        f"Validación I-2026 correcta: hoja {i2026_sheet}, "
        f"{len(i2026_columns)} columnas originales y "
        f"{len(REQUIRED_COLUMNS)} columnas requeridas presentes."
    )
else:
    print(
        "I-2026 no está en data/raw. El notebook puede continuar con 2025, "
        "pero no podrá preparar el Parquet de prueba final."
    )


## Auditoría de esquemas e identificación de períodos

El período se deriva exclusivamente del nombre del archivo. Se preserva
TRIMESTRE como campo original para auditoría; no se transforma en trimestre
calendario. En particular, I-2026 se identifica como 2026T1 aunque su
TRIMESTRE original pueda contener el código 6.


In [ ]:
schema_audit_rows = []

for period, path in sorted(files_by_period.items()):
    sheet_name = "Personas_ENEIC_T1-2026" if period == "2026T1" else 0
    original_columns = read_excel_columns(path, sheet_name)
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(original_columns))

    if missing_columns:
        relevant = [column for column in original_columns if column.startswith(("P02", "P03", "P05"))]
        raise ValueError(
            f"Archivo: {path.name}. Faltan columnas requeridas: {missing_columns}. "
            f"Columnas disponibles relevantes: {relevant[:80]}"
        )

    schema_audit_rows.append(
        {
            "periodo_archivo": period,
            "archivo": path.name,
            "columnas_originales": len(original_columns),
            "columnas_seleccionadas": len(SOURCE_COLUMNS),
            "columnas_faltantes": len(missing_columns),
        }
    )

if schema_audit_rows:
    display(pd.DataFrame(schema_audit_rows).sort_values("periodo_archivo"))


In [ ]:
loaded_by_period = {}
original_record_counts = {}

for period, path in sorted(files_by_period.items()):
    metadata = detect_period_from_filename(path)
    sheet_name = "Personas_ENEIC_T1-2026" if period == "2026T1" else 0

    period_df = load_excel_to_spark(
        path=path,
        sheet_name=sheet_name,
        spark=spark,
        selected_columns=SOURCE_COLUMNS,
        metadata=metadata,
    ).cache()

    original_record_counts[period] = period_df.count()
    loaded_by_period[period] = period_df

    print(f"\n{period} | {path.name} | registros originales: {original_record_counts[period]}")
    period_df.printSchema()
    period_df.select(
        "periodo_archivo",
        "ANIO",
        "TRIMESTRE",
        "edad",
        "salario_mensual",
        "categoria_ocupacional",
    ).show(5, truncate=False)


## Armonización y unión de 2025

El conjunto de 2025 solo se construye si están los cuatro períodos requeridos.
Cada archivo conserva un esquema analítico homogéneo antes de unirse mediante
unionByName.

Una unión por posición sería insegura: IV-2025 tiene una cantidad de columnas
distinta de los otros períodos y el orden de variables puede cambiar. Emparejar
columnas solo por su posición podría asociar variables incorrectas. Por ello,
se seleccionan las columnas analíticas comunes y se utiliza unionByName.


In [ ]:
loaded_periods = set(loaded_by_period)
missing_2025_files = [period for period in EXPECTED_2025 if period not in loaded_periods]
HAS_ALL_2025 = not missing_2025_files

df_2025_raw = None
df_2025_prepared = None
audit_2025 = None
quality_by_period = None

if HAS_ALL_2025:
    ordered_2025 = [loaded_by_period[period] for period in EXPECTED_2025]
    df_2025_raw = ordered_2025[0]

    for period_df in ordered_2025[1:]:
        df_2025_raw = df_2025_raw.unionByName(period_df, allowMissingColumns=False)

    df_2025_raw = df_2025_raw.cache()
    print(
        "Los cuatro períodos de 2025 están disponibles. "
        f"Registros armonizados: {df_2025_raw.count():,}."
    )
else:
    print(
        "No se construye un conjunto parcial de 2025. Faltan: "
        + ", ".join(missing_2025_files)
        + "."
    )


## Preparación aislada de Personas I-2026

I-2026 se armoniza con la misma lógica técnica, pero permanece totalmente
separada. No se mezcla con 2025 ni se utiliza para faltantes, estadística
descriptiva, correlaciones, selección de K o segmentación de este avance.


In [ ]:
df_2026_raw = loaded_by_period.get("2026T1")
df_2026_prepared = None
audit_2026 = None

if df_2026_raw is not None:
    df_2026_prepared, audit_2026 = apply_filters_with_audit(df_2026_raw)
    df_2026_prepared = df_2026_prepared.cache()

    metadata_check = (
        df_2026_prepared.select(
            "archivo_origen",
            "periodo_archivo",
            "anio_archivo",
            "trimestre_calendario",
            "TRIMESTRE",
        )
        .limit(5)
    )
    print("Validación técnica de metadatos para I-2026 (sin análisis empírico):")
    metadata_check.show(truncate=False)
    print("Auditoría técnica de filtros aplicada a I-2026, sin incorporarla al avance 2025:")
    audit_2026.show(truncate=False)
else:
    print("No hay archivo Personas I-2026 para preparar.")


## Faltantes antes de filtros

Los faltantes se calculan sobre el conjunto armonizado de 2025 antes de
excluir registros. La función considera valores nulos, NaN y cadenas vacías o
con espacios cuando corresponda.


In [ ]:
missing_columns = [
    "salario_mensual",
    "edad",
    "antiguedad_anios",
    "antiguedad_meses",
    "horas_semanales",
    "nivel_educativo",
    "categoria_ocupacional",
    "dominio",
    "ocupado",
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR",
]

if HAS_ALL_2025:
    missing_2025 = missing_summary(df_2025_raw, missing_columns)
    missing_2025.orderBy(F.desc("porcentaje_faltante"), F.asc("variable")).show(
        100, truncate=False
    )
else:
    print(
        "Esta tabla requiere los cuatro archivos Personas 2025 y no se calcula "
        "con I-2026."
    )


## Filtros y auditoría

La población analítica incluye personas con edad interpretable de al menos 15
años, ocupadas, en categorías ocupacionales 1–4 y con salario mensual válido
procedente exclusivamente de P05D01. No se imputan salarios ni se eliminan
valores por ser extremos.

La auditoría conserva el efecto de cada filtro, en el mismo orden: edad
interpretable, edad mínima, ocupación, categoría ocupacional, salario
interpretable y positivo/finito, antigüedad válida y compatible con la edad,
y horas semanales entre 1 y 168.


In [ ]:
if HAS_ALL_2025:
    df_2025_prepared, audit_2025 = apply_filters_with_audit(df_2025_raw)
    df_2025_prepared = df_2025_prepared.cache()

    print("Auditoría global de filtros para 2025:")
    audit_2025.show(100, truncate=False)

    originals_by_period = df_2025_raw.groupBy("periodo_archivo").agg(
        F.count("*").alias("registros_originales")
    )
    filtered_by_period = df_2025_prepared.groupBy("periodo_archivo").agg(
        F.count("*").alias("registros_filtrados")
    )

    quality_by_period = (
        originals_by_period.join(filtered_by_period, "periodo_archivo", "left")
        .fillna({"registros_filtrados": 0})
        .withColumn(
            "excluidos",
            F.col("registros_originales") - F.col("registros_filtrados"),
        )
        .withColumn(
            "porcentaje_conservado",
            F.round(
                F.lit(100.0)
                * F.col("registros_filtrados")
                / F.col("registros_originales"),
                2,
            ),
        )
        .orderBy("periodo_archivo")
    )

    print("Resumen de calidad por período:")
    quality_by_period.show(truncate=False)
else:
    print("La auditoría empírica de 2025 queda pendiente de los cuatro archivos.")


## Duplicados

La clave de revisión es periodo_archivo, NUM_HOGAR y NUM_PERSONA. Una persona
en períodos distintos no es duplicada: la unidad observada es persona-período
y puede cambiar a través del tiempo. Por transparencia, no se eliminan
duplicados automáticamente; si aparecen, se clasifican como idénticos o
conflictivos y se presenta una muestra.


In [ ]:
if HAS_ALL_2025:
    duplicate_keys = (
        df_2025_raw.groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
        .agg(F.count("*").alias("n_registros"))
        .where(F.col("n_registros") > 1)
    )

    duplicate_rows = df_2025_raw.join(
        duplicate_keys.select("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"),
        on=["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"],
        how="inner",
    )

    fingerprint_columns = [
        F.coalesce(F.col(column).cast("string"), F.lit("<NULO>"))
        for column in df_2025_raw.columns
    ]
    duplicate_evidence = (
        duplicate_rows.withColumn("_fila_hash", F.sha2(F.concat_ws("||", *fingerprint_columns), 256))
        .groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
        .agg(
            F.count("*").alias("n_registros"),
            F.countDistinct("_fila_hash").alias("filas_distintas"),
        )
        .withColumn(
            "clasificacion",
            F.when(F.col("filas_distintas") == 1, F.lit("idénticos"))
            .otherwise(F.lit("conflictivos")),
        )
        .orderBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
    )

    n_duplicate_keys = duplicate_evidence.count()
    print(f"Claves persona-período repetidas: {n_duplicate_keys:,}")
    if n_duplicate_keys:
        duplicate_evidence.show(20, truncate=False)
        duplicate_rows.orderBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").show(
            20, truncate=False
        )
    else:
        print("No se encontraron claves persona-período repetidas.")
else:
    print("La revisión de duplicados requiere los cuatro archivos de 2025.")


### Respuestas conceptuales de calidad

**¿Por qué IV de 2025 no puede apilarse por posición?**  
IV-2025 puede tener 302 columnas frente a 270 en otros períodos. Además de la
diferencia de cantidad, un cambio de orden o de esquema puede emparejar una
variable con otra incorrecta. Por eso se selecciona un esquema común y se usa
unionByName.

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde
y una respuesta no registrada?**  
El primer caso no aplica por diseño del cuestionario y puede ser esperable para
ciertos perfiles. El segundo es una respuesta esperada que no fue registrada,
por lo que representa un problema de completitud diferente.

**¿Por qué una persona observada en dos períodos no es un duplicado?**  
Cada fila representa una observación persona-período. En un diseño con
observaciones a través del tiempo, la misma persona puede aparecer nuevamente
y sus características pueden haber cambiado.

**¿Por qué los registros filtrados no representan a todos los trabajadores del
país?**  
La base es una muestra de encuesta y se aplican filtros adicionales. Además,
este avance no usa FACTOR para ponderar. Los resultados describen registros
analizados, no un censo ni una estimación oficial para toda Guatemala.

FACTOR es el factor de expansión de la encuesta y permitiría estimaciones
poblacionales que consideren el diseño muestral. Se conserva en los datos, pero
no se usa en las estadísticas principales porque el laboratorio solicita
análisis no ponderados.


## Escritura de Parquet

Los Parquet preparados se guardan por separado. El de 2025 se escribe solo
cuando existen sus cuatro períodos; I-2026 nunca se une a ese conjunto.


In [ ]:
if df_2026_prepared is not None:
    parquet_2026 = DATA_PROCESSED / "eneic_personas_2026T1_preparado.parquet"
    df_2026_prepared.write.mode("overwrite").parquet(str(parquet_2026))
    print(f"I-2026 preparado y guardado por separado en: {parquet_2026}")
else:
    print("No se escribió Parquet de I-2026 porque el archivo no está disponible.")

if HAS_ALL_2025:
    parquet_2025 = DATA_PROCESSED / "eneic_personas_2025_preparado.parquet"
    df_2025_prepared.write.mode("overwrite").parquet(str(parquet_2025))
    print(f"2025 preparado y guardado en: {parquet_2025}")
else:
    print(
        "No se escribe eneic_personas_2025_preparado.parquet: "
        "faltan períodos 2025."
    )


# 2. Estadística descriptiva y exploración

Las siguientes celdas se ejecutan únicamente con la población analítica de
2025. Las estadísticas principales se calculan con Spark. Para gráficas se
transfieren solamente agregados pequeños o muestras con límites explícitos.


In [ ]:
def aggregate_to_pandas(frame, max_rows=1000):
    """Convierte a pandas solamente un agregado final acotado."""
    return frame.limit(max_rows).toPandas()

def bounded_sample_to_pandas(frame, max_rows, seed=SEED):
    """Muestrea en Spark y transfiere como máximo max_rows para graficar."""
    total = frame.count()
    if total == 0:
        return pd.DataFrame(columns=frame.columns)

    fraction = min(1.0, max_rows / total)
    sample = frame.sample(withReplacement=False, fraction=fraction, seed=seed)
    return sample.limit(max_rows).toPandas()

def category_distribution(frame, column):
    category = F.when(
        F.col(column).isNull() | (F.trim(F.col(column).cast("string")) == ""),
        F.lit("DESCONOCIDO"),
    ).otherwise(F.col(column).cast("string"))

    counts = frame.select(category.alias("categoria")).groupBy("categoria").agg(
        F.count("*").alias("n")
    )
    total = frame.count()
    return (
        counts.withColumn("porcentaje", F.round(F.lit(100.0) * F.col("n") / total, 2))
        .orderBy(F.desc("n"), F.asc("categoria"))
    )

def plot_category_distribution(summary, title, x_label):
    plot_df = aggregate_to_pandas(summary.orderBy(F.desc("n"), F.asc("categoria")), 30)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=plot_df, x="categoria", y="porcentaje", color="#3B82F6", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Porcentaje de registros")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
    return plot_df

def group_salary_summary(frame, group_column):
    return (
        frame.groupBy(group_column)
        .agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias(
                "mediana_salario"
            ),
        )
        .orderBy(F.desc("mediana_salario"), F.asc(group_column))
    )

def plot_median_salary(summary, title, x_label):
    plot_df = aggregate_to_pandas(summary, 100)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=plot_df, x=plot_df.columns[0], y="mediana_salario", color="#14B8A6", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Salario mensual mediano")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
    return plot_df


In [ ]:
numeric_variables = [
    "salario_mensual",
    "edad",
    "antiguedad",
    "horas_semanales",
]

if HAS_ALL_2025:
    descriptive_2025 = descriptive_statistics(df_2025_prepared, numeric_variables)
    descriptive_2025.show(100, truncate=False)
else:
    print(
        "Las estadísticas descriptivas requieren los cuatro períodos Personas 2025. "
        "I-2026 no se utiliza como sustituto."
    )


## Distribuciones categóricas

Los conteos y porcentajes se agregan con Spark. Debajo de cada gráfica se
muestra una interpretación descriptiva basada en la categoría más frecuente,
sin atribuir causalidad.


In [ ]:
if HAS_ALL_2025:
    categorical_specs = [
        ("categoria_ocupacional", "Distribución por categoría ocupacional", "Categoría ocupacional"),
        ("nivel_educativo", "Distribución por nivel educativo", "Nivel educativo"),
        ("dominio", "Distribución por dominio", "Dominio"),
    ]

    categorical_summaries = {}
    for column, title, x_label in categorical_specs:
        summary = category_distribution(df_2025_prepared, column)
        categorical_summaries[column] = summary
        plot_category_distribution(summary, title, x_label)

        top_row = summary.first()
        display(
            Markdown(
                f"**Interpretación.** En los registros analizados, la categoría "
                f"más frecuente de {x_label.lower()} fue {top_row['categoria']} "
                f"({top_row['porcentaje']:.2f}% de los registros)."
            )
        )
else:
    print("Las distribuciones categóricas quedan pendientes de 2025 completo.")


## Salario mensual

La variable salarial es P05D01, renombrada como salario_mensual. No se aplica
recorte por IQR, winsorización, capping ni transformación del salario para los
cálculos obligatorios. La escala logarítmica siguiente se emplea solo para
visualizar la cola de la distribución.


In [ ]:
if HAS_ALL_2025:
    salary_sample = bounded_sample_to_pandas(
        df_2025_prepared.select("salario_mensual"), max_rows=10_000, seed=SEED
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(salary_sample["salario_mensual"], bins=40, color="#2563EB", edgecolor="white")
    axes[0].set_title("Histograma de salario mensual")
    axes[0].set_xlabel("Salario mensual")
    axes[0].set_ylabel("Frecuencia en muestra")

    axes[1].hist(salary_sample["salario_mensual"], bins=40, color="#7C3AED", edgecolor="white")
    axes[1].set_xscale("log")
    axes[1].set_title("Salario mensual con escala logarítmica")
    axes[1].set_xlabel("Salario mensual (escala logarítmica, solo visual)")
    axes[1].set_ylabel("Frecuencia en muestra")
    plt.tight_layout()
    plt.show()

    salary_center = df_2025_prepared.agg(
        F.avg("salario_mensual").alias("media"),
        F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias("mediana"),
    ).first()

    mean_salary = float(salary_center["media"])
    median_salary = float(salary_center["mediana"])
    difference = mean_salary - median_salary

    display(
        Markdown(
            f"**Interpretación.** En todos los registros elegibles de 2025, "
            f"la media salarial fue {mean_salary:,.2f} y la mediana fue "
            f"{median_salary:,.2f}. La diferencia media menos mediana fue "
            f"{difference:,.2f}; esta comparación describe la asimetría "
            f"observada sin eliminar salarios extremos."
        )
    )
else:
    print("Los histogramas y medidas salariales requieren 2025 completo.")


## Salario por educación y categoría ocupacional

Las medianas se calculan en Spark para evitar que el conjunto completo pase a
pandas. Las comparaciones describen diferencias observadas entre grupos y no
establecen relaciones causales.


In [ ]:
if HAS_ALL_2025:
    salary_by_education = group_salary_summary(df_2025_prepared, "nivel_educativo")
    salary_by_education.show(100, truncate=False)
    education_plot = plot_median_salary(
        salary_by_education,
        "Mediana de salario por nivel educativo",
        "Nivel educativo",
    )

    top_education = salary_by_education.first()
    display(
        Markdown(
            f"**Interpretación.** La mayor mediana observada por nivel educativo "
            f"correspondió al código {top_education['nivel_educativo']} "
            f"({top_education['mediana_salario']:,.2f}), con "
            f"{top_education['n']:,} registros. Esta comparación no implica causalidad."
        )
    )

    salary_by_category = group_salary_summary(df_2025_prepared, "categoria_ocupacional")
    salary_by_category.show(100, truncate=False)
    category_plot = plot_median_salary(
        salary_by_category,
        "Mediana de salario por categoría ocupacional",
        "Categoría ocupacional",
    )

    top_category = salary_by_category.first()
    display(
        Markdown(
            f"**Interpretación.** La mayor mediana observada por categoría "
            f"ocupacional correspondió a {top_category['categoria_ocupacional']} "
            f"({top_category['mediana_salario']:,.2f}), con "
            f"{top_category['n']:,} registros. El resultado es descriptivo."
        )
    )
else:
    print("Los resúmenes salariales por grupo requieren 2025 completo.")


## Evolución por trimestre

La siguiente gráfica utiliza solamente los cuatro períodos de 2025. Resume el
tamaño de muestra analítico y la mediana salarial de cada período; I-2026 no
participa.


In [ ]:
if HAS_ALL_2025:
    temporal_2025 = (
        df_2025_prepared.groupBy("periodo_archivo")
        .agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias(
                "salario_mediano"
            ),
        )
        .orderBy("periodo_archivo")
    )
    temporal_2025.show(truncate=False)

    temporal_plot = aggregate_to_pandas(temporal_2025, 10)
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(temporal_plot["periodo_archivo"], temporal_plot["n"], marker="o", color="#2563EB")
    ax1.set_xlabel("Período")
    ax1.set_ylabel("Registros analíticos", color="#2563EB")
    ax1.tick_params(axis="y", labelcolor="#2563EB")

    ax2 = ax1.twinx()
    ax2.plot(
        temporal_plot["periodo_archivo"],
        temporal_plot["salario_mediano"],
        marker="s",
        color="#DC2626",
    )
    ax2.set_ylabel("Salario mensual mediano", color="#DC2626")
    ax2.tick_params(axis="y", labelcolor="#DC2626")
    ax1.set_title("Tamaño de muestra y salario mediano por trimestre de 2025")
    plt.tight_layout()
    plt.show()

    first_period = temporal_plot.iloc[0]
    last_period = temporal_plot.iloc[-1]
    display(
        Markdown(
            f"**Interpretación.** Entre {first_period['periodo_archivo']} y "
            f"{last_period['periodo_archivo']}, el tamaño de muestra analítico "
            f"cambió de {int(first_period['n']):,} a {int(last_period['n']):,}; "
            f"la mediana salarial cambió de {first_period['salario_mediano']:,.2f} "
            f"a {last_period['salario_mediano']:,.2f}."
        )
    )
else:
    print("La evolución temporal requiere los cuatro períodos de 2025.")


# 3. Relaciones entre variables numéricas

Se evalúan correlaciones de Pearson con todos los registros elegibles de 2025
y con el componente Correlation de pyspark.ml. Una correlación mide asociación
lineal; no demuestra causalidad.


In [ ]:
CORRELATION_COLUMNS = [
    "salario_mensual",
    "edad",
    "antiguedad",
    "horas_semanales",
]
CORRELATION_LABELS = [
    "Salario mensual",
    "Edad",
    "Antigüedad",
    "Horas semanales",
]

corr_matrix = None
strongest_salary_association = None
age_tenure_correlation = None

if HAS_ALL_2025:
    corr_assembler = VectorAssembler(
        inputCols=[
            "salario_mensual",
            "edad",
            "antiguedad",
            "horas_semanales",
        ],
        outputCol="corr_features",
        handleInvalid="skip",
    )
    corr_dataframe = corr_assembler.transform(df_2025_prepared)

    corr_result = Correlation.corr(
        corr_dataframe,
        "corr_features",
        "pearson",
    ).head()

    corr_matrix = corr_result[0].toArray()
    corr_table = pd.DataFrame(
        corr_matrix,
        index=CORRELATION_LABELS,
        columns=CORRELATION_LABELS,
    )
    display(corr_table)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        corr_table,
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        ax=ax,
    )
    ax.set_title("Matriz de correlaciones de Pearson, 2025")
    plt.tight_layout()
    plt.show()

    salary_pairs = [
        (CORRELATION_LABELS[index], float(corr_matrix[0, index]))
        for index in range(1, len(CORRELATION_LABELS))
        if math.isfinite(float(corr_matrix[0, index]))
    ]
    if salary_pairs:
        strongest_salary_association = max(salary_pairs, key=lambda item: abs(item[1]))
        display(
            Markdown(
                f"**Interpretación.** La mayor asociación lineal con salario "
                f"mensual fue {strongest_salary_association[0]} con "
                f"r = {strongest_salary_association[1]:.3f}. "
                f"El valor describe asociación, no causalidad."
            )
        )
    else:
        display(Markdown("No fue posible identificar una correlación salarial finita."))

    age_tenure_correlation = float(corr_matrix[1, 2])
    if math.isfinite(age_tenure_correlation):
        display(
            Markdown(
                f"**Edad y antigüedad.** El coeficiente de Pearson fue "
                f"r = {age_tenure_correlation:.3f}. "
                f"Este resultado resume la relación lineal observada."
            )
        )
    else:
        display(Markdown("La correlación entre edad y antigüedad no fue finita."))
else:
    print("La matriz de correlaciones requiere los cuatro períodos de 2025.")


# 4. Segmentación de perfiles mediante KMeans

Los perfiles se construyen con edad, antigüedad y horas semanales. El salario
no se incluye inicialmente en el vector de clustering: se utiliza después para
describir los segmentos. Esto evita que su dispersión condicione directamente
la creación de perfiles personales y laborales.


In [ ]:
best_k = None
best_predictions = None
silhouette_results = pd.DataFrame(columns=["k", "silhouette", "estado"])
cluster_characterization = None

if HAS_ALL_2025:
    kmeans_assembler = VectorAssembler(
        inputCols=["edad", "antiguedad", "horas_semanales"],
        outputCol="features_unscaled",
        handleInvalid="skip",
    )
    features_unscaled = kmeans_assembler.transform(df_2025_prepared)

    scaler = StandardScaler(
        inputCol="features_unscaled",
        outputCol="features",
        withMean=True,
        withStd=True,
    )
    scaler_model = scaler.fit(features_unscaled)
    kmeans_features = scaler_model.transform(features_unscaled).cache()

    feature_rows = kmeans_features.count()
    variance_row = df_2025_prepared.agg(
        F.variance("edad").alias("var_edad"),
        F.variance("antiguedad").alias("var_antiguedad"),
        F.variance("horas_semanales").alias("var_horas"),
    ).first()
    zero_variance = [
        name
        for name, value in variance_row.asDict().items()
        if value is None or float(value) == 0.0
    ]

    if feature_rows < 5 or zero_variance:
        print(
            "No se puede ejecutar KMeans: se requieren al menos cinco registros "
            f"y varianza positiva en las tres variables. Variables sin varianza: "
            f"{zero_variance}."
        )
    else:
        kmeans_models = {}
        predictions_by_k = {}
        score_rows = []

        for k in (2, 3, 4, 5):
            try:
                model = KMeans(
                    k=k,
                    seed=SEED,
                    featuresCol="features",
                    predictionCol="prediction",
                ).fit(kmeans_features)
                predicted = model.transform(kmeans_features).cache()

                evaluator = ClusteringEvaluator(
                    predictionCol="prediction",
                    featuresCol="features",
                    metricName="silhouette",
                )
                silhouette = float(evaluator.evaluate(predicted))

                kmeans_models[k] = model
                predictions_by_k[k] = predicted
                score_rows.append({"k": k, "silhouette": silhouette, "estado": "calculado"})
            except Exception as error:
                score_rows.append(
                    {"k": k, "silhouette": None, "estado": f"no calculado: {error}"}
                )

        silhouette_results = pd.DataFrame(score_rows).sort_values("k")
        display(silhouette_results)

        valid_scores = silhouette_results.dropna(subset=["silhouette"]).sort_values(
            ["silhouette", "k"],
            ascending=[False, True],
        )
        if not valid_scores.empty:
            best_k = int(valid_scores.iloc[0]["k"])
            best_predictions = predictions_by_k[best_k]
            print(
                f"K seleccionado: {best_k}; silhouette: "
                f"{float(valid_scores.iloc[0]['silhouette']):.4f}."
            )
        else:
            print("No se obtuvo una silhouette válida para K entre 2 y 5.")
else:
    print("KMeans requiere los cuatro períodos Personas 2025.")


## Selección de K

Se prueban K = 2, 3, 4 y 5 con semilla 42. La selección se basa en la mayor
silhouette disponible; si hubiera empate exacto, se prefiere el K menor. No se
predefine el número ganador.


In [ ]:
if HAS_ALL_2025 and not silhouette_results.empty:
    plot_scores = silhouette_results.dropna(subset=["silhouette"])
    if not plot_scores.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(plot_scores["k"], plot_scores["silhouette"], marker="o", color="#7C3AED")
        ax.set_title("K versus silhouette")
        ax.set_xlabel("Número de clusters (K)")
        ax.set_ylabel("Silhouette")
        ax.set_xticks([2, 3, 4, 5])
        plt.tight_layout()
        plt.show()

        display(
            Markdown(
                f"**Decisión.** Se seleccionó K = {best_k} por tener la mayor "
                f"silhouette calculada ({float(plot_scores.loc[plot_scores['k'] == best_k, 'silhouette'].iloc[0]):.4f})."
            )
        )
    else:
        print("No hay valores de silhouette para graficar.")
else:
    print("La selección de K queda pendiente de 2025 completo.")


## Caracterización de clusters

Las medidas siguientes se calculan sobre todos los registros de cada cluster.
Las categorías se resumen como la modalidad predominante por cluster para
mantener tablas compactas.


In [ ]:
def predominant_category_by_cluster(predictions, column):
    categorical = predictions.withColumn(
        "_categoria",
        F.when(
            F.col(column).isNull() | (F.trim(F.col(column).cast("string")) == ""),
            F.lit("DESCONOCIDO"),
        ).otherwise(F.col(column).cast("string")),
    )
    totals = categorical.groupBy("prediction").agg(F.count("*").alias("cluster_n"))
    counts = categorical.groupBy("prediction", "_categoria").agg(F.count("*").alias("n"))
    window = Window.partitionBy("prediction").orderBy(F.desc("n"), F.asc("_categoria"))

    return (
        counts.join(totals, "prediction")
        .withColumn("porcentaje", F.round(F.lit(100.0) * F.col("n") / F.col("cluster_n"), 2))
        .withColumn("rango", F.row_number().over(window))
        .where(F.col("rango") == 1)
        .select(
            "prediction",
            F.col("_categoria").alias(column),
            "n",
            "porcentaje",
        )
        .orderBy("prediction")
    )

if best_predictions is not None:
    total_cluster_records = best_predictions.count()
    cluster_characterization = (
        best_predictions.groupBy("prediction")
        .agg(
            F.count("*").alias("n"),
            F.avg("edad").alias("edad_media"),
            F.expr("percentile_approx(edad, 0.5, 10000)").alias("edad_mediana"),
            F.avg("antiguedad").alias("antiguedad_media"),
            F.expr("percentile_approx(antiguedad, 0.5, 10000)").alias(
                "antiguedad_mediana"
            ),
            F.avg("horas_semanales").alias("horas_media"),
            F.expr("percentile_approx(horas_semanales, 0.5, 10000)").alias(
                "horas_mediana"
            ),
            F.avg("salario_mensual").alias("salario_media"),
            F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias(
                "salario_mediana"
            ),
        )
        .withColumn(
            "porcentaje",
            F.round(F.lit(100.0) * F.col("n") / F.lit(total_cluster_records), 2),
        )
        .orderBy("prediction")
    )
    cluster_characterization.show(100, truncate=False)

    for category_column in ("nivel_educativo", "categoria_ocupacional", "dominio"):
        print(f"Categoría predominante: {category_column}")
        predominant_category_by_cluster(best_predictions, category_column).show(
            100, truncate=False
        )

    overall_medians = df_2025_prepared.agg(
        F.expr("percentile_approx(edad, 0.5, 10000)").alias("edad"),
        F.expr("percentile_approx(antiguedad, 0.5, 10000)").alias("antiguedad"),
        F.expr("percentile_approx(horas_semanales, 0.5, 10000)").alias("horas"),
    ).first().asDict()

    cluster_lines = []
    for row in cluster_characterization.collect():
        age_phrase = "edad relativamente menor" if row["edad_mediana"] < overall_medians["edad"] else "edad relativamente mayor"
        tenure_phrase = "menor antigüedad" if row["antiguedad_mediana"] < overall_medians["antiguedad"] else "mayor antigüedad"
        hours_phrase = "menos horas semanales" if row["horas_mediana"] < overall_medians["horas"] else "más horas semanales"
        cluster_lines.append(
            f"- Cluster {row['prediction']}: {age_phrase}, {tenure_phrase} y "
            f"{hours_phrase}; descripción basada en sus medianas observadas."
        )

    display(Markdown("**Perfiles interpretativos posteriores a los resultados**\n" + "\n".join(cluster_lines)))
else:
    print("La caracterización queda pendiente de una selección válida de K.")


## Visualizaciones de clusters

Los diagramas de dispersión usan una muestra de como máximo 5,000 registros con
semilla 42. Las estadísticas de caracterización anteriores usan el conjunto
completo de cada cluster.


In [ ]:
if best_predictions is not None:
    scatter_sample = bounded_sample_to_pandas(
        best_predictions.select(
            "edad",
            "antiguedad",
            "horas_semanales",
            "prediction",
        ),
        max_rows=5_000,
        seed=SEED,
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(
        data=scatter_sample,
        x="edad",
        y="antiguedad",
        hue="prediction",
        palette="tab10",
        alpha=0.65,
        s=24,
        ax=axes[0],
    )
    axes[0].set_title("Edad versus antigüedad por cluster")
    axes[0].set_xlabel("Edad (años)")
    axes[0].set_ylabel("Antigüedad (años)")

    sns.scatterplot(
        data=scatter_sample,
        x="antiguedad",
        y="horas_semanales",
        hue="prediction",
        palette="tab10",
        alpha=0.65,
        s=24,
        ax=axes[1],
    )
    axes[1].set_title("Antigüedad versus horas semanales por cluster")
    axes[1].set_xlabel("Antigüedad (años)")
    axes[1].set_ylabel("Horas semanales")
    plt.tight_layout()
    plt.show()
else:
    print("Las visualizaciones de clusters requieren una solución KMeans válida.")


# Conclusiones del avance


In [ ]:
if HAS_ALL_2025 and df_2025_prepared is not None:
    original_n = df_2025_raw.count()
    filtered_n = df_2025_prepared.count()
    retained_pct = 100.0 * filtered_n / original_n if original_n else float("nan")

    conclusions = [
        (
            f"Se armonizaron los cuatro períodos de Personas 2025 con "
            f"{original_n:,} registros originales y {filtered_n:,} registros "
            f"analíticos ({retained_pct:.2f}% conservado tras las validaciones)."
        ),
        (
            f"La distribución salarial se describió sin recortar extremos; la "
            f"media fue {mean_salary:,.2f} y la mediana {median_salary:,.2f}."
        ),
    ]

    if strongest_salary_association is not None:
        conclusions.append(
            f"La mayor asociación lineal observada con salario fue "
            f"{strongest_salary_association[0]} "
            f"(r = {strongest_salary_association[1]:.3f}); no se interpreta "
            f"como causalidad."
        )

    if best_k is not None:
        winning_silhouette = float(
            silhouette_results.loc[
                silhouette_results["k"] == best_k, "silhouette"
            ].iloc[0]
        )
        conclusions.append(
            f"La segmentación KMeans seleccionó K = {best_k} con silhouette "
            f"{winning_silhouette:.4f}; los perfiles se definieron con edad, "
            f"antigüedad y horas semanales, y el salario se usó solo para describirlos."
        )

    conclusions.append(
        "FACTOR se conservó, pero el análisis es no ponderado: las conclusiones "
        "describen los registros analizados y no estimaciones oficiales de toda Guatemala."
    )

    display(Markdown("\n".join(f"{index + 1}. {text}" for index, text in enumerate(conclusions))))
else:
    display(
        Markdown(
            "Las conclusiones empíricas se completarán una vez estén disponibles "
            "los cuatro períodos de 2025."
        )
    )
